# INF201 - Deliverable 3
## *Game Simulation*

**Group:** G6_2: Nikolai Friis, Tommy Rustad Apelseth, Aron Lerdahl

**GitHub repository:** https://github.com/tommyra06/INF201-Deliverables

---

### Contents
* Setup
* Task 1: *Update the Character class*
* Task 2: *Introducing randomness with a Dice class*
* Task 3: *Create items*
* Task 4: *Character specialization*
* Task 5: *Running battles*
* Use of generative AI

---

## Setup

### Virtual environment

Create a new virtual environment (`.venv`). Install the packages for Deliverable 3, listed in `requirements.txt`. Use the command pallete.

### Input data
*TODO: Describe where the input data comes from (e.g. a script in `Oppgavefiler/`), where it is saved, and which files it creates.*

* `TODO` - *description*
* `TODO` - *description*

*TODO: Note if the data folder is not uploaded to GitHub and must be generated before running this notebook.*

### Imports (Library)
All imports used in the notebook are collected in the cell below.

In [5]:
# Library imports that come with Python
import random # Random numbers for the dice role in the Dice class (Used for task 2)

# Installed libraries, listed in `requirements.txt`
# None are used in this notebook

---

## Task 1:  Update the Character class

For Task 1, we have updated the character class from the lecture with two additional methods and one property, we have also updated the attack function.

* `take_damage` - Method that takes parameter `dmg_taken` and subtracts said value from `_hp`, uses the `max` function to make sure the `_hp` is not under 0. 
* `attack` - Method that activates `take_damage` method in target class.
* `heal` - Method that increases `_hp` and uses the `min` function to make sure `_hp` isn't over `hp_max`.
* `is_alive` - Property that returns bool based of if `_hp` is above or below 0.


In [6]:
class Character:
    """Represents a character in the RPG"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon = None) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage delt when character attacks (default = 10)
            weapon: Optional weapon (default = None)
        """
        
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.weapon = weapon
        

    def attack(self, target) -> None:
        """Activates take_damage function for target"""
        if self.weapon != None:
            target.take_damage(self.weapon.roll_damage())
        else:
            target.take_damage(self.damage)
        

    def take_damage(self, dmg_taken: int) -> None:
        """Lowers hp based of damage taken"""
        self._hp -= dmg_taken
        self._hp = max(self._hp, 0)
        
    def heal(self) -> None:
        """Increases hp"""
        self._hp += 10
        self._hp = min(self._hp, self.hp_max)

    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>)"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"


    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        """Checks if hp is below 0 and returns alive status"""
        return self._hp > 0

---

### Testing the relevant methods

Some simple tests can show us the results:

* One character attacks another, which should lose 10 HP (100 → 90).
* The injured character heals, which should bring it back to 100 HP.
* Healing at full HP should not go above `hp_max` (stays at 100).
* Taking 150 damage should stop at 0 HP, not go negative, and `is_alive` should become False.
* Trying to change `is_alive` directly should give an error, since it is a read-only property.

---

In [ ]:
hero = Character("Hero")
enemy = Character("Enemy")

# attack calls the target's take_damage, the enemy should lose 10 HP
hero.attack(enemy)
print("Enemy HP after one attack:", enemy.hp)

# heal increases HP, but never above hp_max
enemy.heal()
print("Enemy HP after healing:", enemy.hp)
enemy.heal()
print("Enemy HP after healing at full HP:", enemy.hp)

# take_damage stops at 0 HP, and is_alive becomes False
enemy.take_damage(150)
print("Enemy HP after 150 damage:", enemy.hp, "| alive:", enemy.is_alive)

# is_alive is read-only
try:
    enemy.is_alive = True
except AttributeError:
    print("is_alive can not be changed directly")

---

### Results and discussion of Task 1

All the results match what we expect.

The attack took exactly 10 HP from the enemy, which shows that `attack` now works through the target's own `take_damage` method instead of changing the target's `_hp` directly. Healing brought the enemy back to 100 HP, and healing again at full HP did not go above `hp_max`, because `heal` uses `min`. Taking 150 damage stopped at 0 HP instead of going negative, because `take_damage` uses `max`, and `is_alive` then returned False.

Trying to set `is_alive` directly gave an error, which shows that it is a read-only property. The only way to change a character's HP is through its own methods, which is the data encapsulation the task asks for.

One limitation is that `heal` always heals a fixed amount of 10 HP. It could be improved by taking the amount as a parameter.

---

## Task 2: *Creating a dice-roll class*

For Task 2, we are writing a dice-roll class with 4 methods.

1. `__init__` - *Initializes the die, with a customizable number of sides. The default is 6.*
2. `roll` - *Uses random to deicide on which of the sides land face-up.*
3. `__repr__` *Returns a string representation of the object.*
4. `__call__` *Lets us call an object as if it were a function.*

---

In [7]:
class Dice:
    """
    Represents a dice that can be rolled to get a random number n>=2, with a default value of 6.
    """

    def __init__(self, sides: int = 6) -> None:
        """
        Create a Dice with the specified number of sides, defaulting to 6.
        """
        
        if sides >= 2:
            self.sides = sides
        else:
            raise ValueError("Dice must have at least 2 sides.")

    def roll(self) -> int:
        """
        Rolls the dice and returns a random number between 1 and the number of sides.
        """
        
        return random.randint(1, self.sides)
    
    def __repr__(self) -> str:
        """
        Returns a string representation of the Dice object.
        """
        
        return f"Dice(sides={self.sides})"
    
    def __call__(self) -> int:
        """
        Allows the Dice object to be called like a function to roll it.
        """
        
        return self.roll()

---

### Testing the relevant methods

*Some simple code to test the different methods to confirm they work before using the class in the full workflow.*

In [8]:
# Test the initialization of the Dice class
d6 = Dice()
d20 = Dice(20)

# Test the representation method
print(repr(d6), repr(d20))

# Test the roll method
print([d6.roll() for _ in range(10)])

# Test the call method
print([d20() for _ in range(10)])

# Testing the value error (uncomment to see the error)
# Dice(1)

Dice(sides=6) Dice(sides=20)
[6, 2, 4, 3, 1, 3, 4, 1, 4, 4]
[9, 20, 8, 11, 18, 15, 6, 13, 13, 14]


---

### Results and discussion of Task 2

All the results match what we expect, and all four methods were tested sucessfully.

We start by testing `__init__`, and the code returns no errors when left at the default of 6 or when making a d20.
The `__repr__` method also sucessfully prints the f-string described in the script, showing us the name of the class and a clear count of the sides.
A simple for loop tests the `roll` method 10 times, and we recieve a random assortment of numbers. There are no indications that the `random` library has any issues with generating the random numbers.
The `call` method is also tested with the same loop, though the die has more sides. Same results as above, `call` uses `roll` and the numbers are as expected.
Finally we make sure a ValueError is raised if a die is initialized with sides <2.



---

## Task 3: Create items

For Task 3, we are writing a `Weapon` class and connecting it to the `Character` class.

1. `Weapon` - A class with a `name` and a `damage_dice`, that uses the `Dice` class from Task 2 to decide how much damage it deals.
2. `Character.attack` - Updated so that a character with an equipped weapon rolls damage from the weapon instead of using its base damage.

The class belongs in the module `items.py` rather than `weapons.py`, so it can later be extended with other items the characters can hold. Weapons are used by the subclasses in Task 4 and in the battles in Task 5.

---

### 3.1 Weapon

The class takes two arguments, `name` and `damage_dice`.

* `name` - The name of the weapon, e.g. `"Sword"`.
* `damage_dice` - A `Dice` object from Task 2 that decides the damage.
* `roll_damage` - Method that calls `self.damage_dice.roll()` and returns the result as an `int`. A weapon with `Dice(8)` therefore deals between 1 and 8 damage.
* `__repr__` - Returns a string representation like `Weapon(name='Sword', damage_dice=Dice(sides=8))`.

This is an example of object composition: a `Weapon` *has* a `Dice`.

In [9]:
class Weapon:
    """Represents a weapon that uses a Dice to decide how much damage it deals"""

    def __init__(self, name: str, damage_dice: Dice) -> None:
        """
        Initialise the Weapon

        Args:
            name (str): The name of the weapon.
            damage_dice (Dice): The dice rolled to decide the damage dealt.
        """
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self) -> int:
        """Rolls the damage dice and returns the damage dealt by the weapon"""
        return self.damage_dice.roll()

    def __repr__(self) -> str:
        """Representation of weapon object"""
        return f"Weapon(name={self.name!r}, damage_dice={self.damage_dice!r})"

---

### 3.2 Equipping characters with weapons

The weapon is connected to the `Character` class from Task 1, so there is only one version of the class in the notebook.

* `Character.__init__` takes an optional argument `weapon`, with `None` as default (no weapon). It is stored in `self.weapon`.
* `attack` checks if the character has a weapon. If it does, the damage is `self.weapon.roll_damage()`. If not, the character uses its base `damage`.
* The damage is passed on to the target's `take_damage` method, so the attacker never changes the target's `_hp` directly.

The subclasses `Warrior` and `Mage` in Task 4 also take a `weapon` argument and pass it on to `Character` with `super()`.

*Below we show how characters are equipped with weapons, either when they are created or afterwards.*

In [10]:
# Create two weapons with different dice
sword = Weapon("Sword", Dice(8))
greataxe = Weapon("Greataxe", Dice(20))

# Characters can be created with or without a weapon
knight = Character("Knight", weapon=sword)
peasant = Character("Peasant")
print(knight.name, "has", knight.weapon)
print(peasant.name, "has", peasant.weapon)

# A weapon can also be equipped after the character is created
peasant.weapon = greataxe
print(peasant.name, "now has", peasant.weapon)

Knight has Weapon(name='Sword', damage_dice=Dice(sides=8))
Peasant has None
Peasant now has Weapon(name='Greataxe', damage_dice=Dice(sides=20))


---

### Testing the relevant methods

Some simple tests where the result is easy to check by hand:

* We create a sword with an 8-sided dice and print it, to test `__repr__`.
* We roll the sword's damage 1000 times and check that the lowest and highest rolls are 1 and 8.
* A character without a weapon attacks a target, which should lose exactly 10 HP (100 → 90).
* A character with the sword attacks a target, which should lose between 1 and 8 HP (100 → 92-99).

In [11]:
# Test the initialization and representation of the Weapon class
sword = Weapon("Sword", Dice(8))
print(repr(sword))

# Test the roll_damage method, all rolls should be between 1 and 8
rolls = [sword.roll_damage() for _ in range(1000)]
print("Lowest roll:", min(rolls), "| Highest roll:", max(rolls))

# Test attack without a weapon, the target should lose exactly 10 HP
peasant = Character("Peasant")
target = Character("Target")
peasant.attack(target)
print("Target HP after unarmed attack:", target.hp)

# Test attack with a weapon, the target should lose between 1 and 8 HP
knight = Character("Knight", weapon=sword)
target = Character("Target")
knight.attack(target)
print("Target HP after sword attack:", target.hp)

Weapon(name='Sword', damage_dice=Dice(sides=8))
Lowest roll: 1 | Highest roll: 8
Target HP after unarmed attack: 90
Target HP after sword attack: 97


---

### Results and discussion of Task 3

All the results match what we expect.

The `__repr__` method prints the name of the weapon and the dice it uses. Over 1000 rolls the lowest damage was 1 and the highest was 8, so `roll_damage` stays within the range of the dice. The character without a weapon dealt exactly 10 damage, while the character with the sword dealt a random amount between 1 and 8, which shows that `attack` uses the weapon when one is equipped.

A weapon is not always better than no weapon. A sword with an 8-sided dice deals 4.5 damage on average, which is less than the base damage of 10. Against a `Warrior` with 5 armor (Task 4), most sword hits deal little or no damage at all. Stronger weapons need a dice with more sides, e.g. `Dice(20)`. Since weapon damage is random, battles with weapons will also give different results each time.

---

## Task 4: Character specialization

For Task 4, we have two subclasses based of off the `Character` class.

1. `Warrior` - Subclass with `armor` that negates incoming `damage`
2. `Mage` - Subclass with `mana` that increases `damage` if `mana` is high enough.



---

### 4.1 Warrior

Subclass has an additional argument for `armor`.
* Overwrites take_damage method by subtracting `dmg_taken` with `armor`, then uses `max` function to make sure `dmg_taken` isn't negative.

In [12]:
class Warrior(Character):
    """Represents the warrior subclass"""
    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon = None, armor: int = 5) -> None:
        super().__init__(name, hp_max, damage, weapon)
        """
            Initialise the Character
    
            Args:
                name (str): The name of the character.
                hp_max (int): Max hit points (default = 100)
                weapon: Optional weapon (default = None)
                armor (int): Damage negated when character is attacked (default = 5)
        """
        self.armor = armor
    

    def take_damage(self, dmg_taken: int) -> None:
        """Lowers hp based of damage taken minus armor"""
        dmg_taken -= self.armor
        dmg_taken = max(dmg_taken, 0)
        return super().take_damage(dmg_taken)

---

### 4.2 Mage

Subclass with two additional arguments, `mana` and `mana_max`.

* Overwrites the `attack` method so if `mana` is equal to or above 10, it will double base damage.
* If `mana` is below 10, it will instead deal the normal amount of damage and increase `mana`.

In [13]:
class Mage(Character):
    """Represents the mage subclass"""
    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon = None, mana: int = 30, mana_max: int = 30) -> None:
        super().__init__(name, hp_max, damage, weapon)
        """
            Initialise the Character
            
            Args:
                name (str): The name of the character.
                hp_max (int): Max hit points (default = 100)
                damage (int): Damage delt when character attacks (default = 10)
                weapon: Optional weapon (default = None)
                mana (int): Amount of mana (default = 30)
                mana_max (int): Max amount of mana (default = 30)        
        """
        self.mana = mana
        self.mana_max = mana_max

    def attack(self, target) -> None:
        """Deals different amount of damage based off of mana left"""

        base_dmg = self.damage
        saved_weapon = self.weapon

        if self.mana >= 10:
            self.mana -= 10
            self.weapon = None
            self.damage *= 2
        else:
            self.mana += 5
            self.mana = min(self.mana, self.mana_max)

        try:
            return super().attack(target)
        
        finally:
            self.damage = base_dmg
            self.weapon = saved_weapon

---

### Testing the relevant methods

Some simple tests where the results can easily be checked:

* A `Warrior` with 5 armor is hit once for 10 damage and once for 3 damage. It should lose 5 HP from the first hit and 0 HP from the second, ending at 95 HP.
* A `Mage` with a staff attacks five times. The first three attacks should be spells dealing 20 damage without using the staff, and the last two should be normal attacks using the staff (1-6 damage) while restoring mana.

In [ ]:
# Warrior: armor reduces a 10 damage hit to 5, and a 3 damage hit to 0
warrior = Warrior("Warrior")
Character("Attacker").attack(warrior)
Character("Weak", damage=3).attack(warrior)
print("Warrior HP:", warrior.hp)  # expected 95

# Mage: 3 spells (20 damage), then normal attacks with the staff that restore mana
mage = Mage("Mage", weapon=Weapon("Staff", Dice(6)))
for i in range(5):
    target = Character("Target")
    mage.attack(target)
    print(f"Attack {i + 1}: {100 - target.hp} damage, {mage.mana} mana left")

### Results and discussion of Task 4

All the results match what we expect.

**Warrior:** The warrior ended with 95 HP. The 10 damage hit was reduced to 5 by the armor, and the 3 damage hit was reduced to 0. Because `take_damage` uses `max` to stop the damage from going below 0, a weak attack can not accidentally heal the warrior.

**Mage:** The mage started with 30 mana and cast three spells in a row, each dealing exactly 20 damage (double the base damage) and costing 10 mana. The spells ignored the staff, which shows that magic attacks do not use the weapon. With 0 mana left, the mage fell back to normal attacks, which used the staff's dice (1-6 damage) and restored 5 mana each time. After two normal attacks the mana is back at 10, so in a long battle the mage will alternate between spells and normal attacks once the starting mana is used up.

The mage is strongest at the start of a battle while it has mana, while the warrior gets more value from its armor the longer the battle lasts.

---

## Task 5: *TODO: Task 5 title* ARON DO TS

For Task 5, we are writing 2 functions

1. `task5_function1` - *TODO: what the function does.*
2. `task5_function2` - *TODO: what the function does.*

*TODO: Short note on how the functions are used later (e.g. in the full workflow) and any requirements from the assignment.*

---

### 5.1 *TODO: Subtask title*

*TODO: Explain what the function takes as input and what it does, step by step:*

* *TODO: Step 1*
* *TODO: Step 2*
* *TODO: Step 3*

*TODO: Explain what the function returns.*

In [14]:
def task5_function1(param: str) -> None:
    """
    TODO: One-line summary of what the function does.

    Parameters
    ----------
    param : str
        TODO: Description of the parameter.

    Returns
    -------
    None
        TODO: Description of the return value.
    """
    ...

---

### 5.2 *TODO: Subtask title*

*TODO: Explain what the function takes as input and what it does, step by step:*

* *TODO: Step 1*
* *TODO: Step 2*
* *TODO: Step 3*

*TODO: Explain what the function returns.*

In [15]:
def task5_function2(param: str) -> None:
    """
    TODO: One-line summary of what the function does.

    Parameters
    ----------
    param : str
        TODO: Description of the parameter.

    Returns
    -------
    None
        TODO: Description of the return value.
    """
    ...

---

### Testing each function

*TODO: Describe what the functions are tested on (generated data or small hand-made examples where the result is easy to check by hand).*

In [16]:
# TODO: Test the Task 5 functions

---

### Results and discussion of Task 5

*TODO: What the tests showed, whether the results match what we expected, and anything worth noting (edge cases, limitations, assumptions).*

---

# Use of Generative AI

### Task 1

* I used chat gpt to orient myself around properties in classes.



---

### Task 2

**Questions about the code**
* Asked Claude to explain the Task 2 requirements: what each part of the `Dice` class should do and how it fits with the `Weapon` class in Task 3.
* Asked how to build a class that meets the requirements.

**Code written by AI**
* No method bodies were written by AI. Claude gave an outline of the class (the four method signatures with short docstrings) with hint comments instead of code.
* The test cell is based on a short example test Claude suggested (`repr`, `roll()`, calling the object, and `Dice(1)` raising an error).

**Testing and discussion**
* Ran the tests in the notebook. The repr matched `Dice(sides=6)`/`Dice(sides=20)` and all rolls stayed within range.

**Errors and corrections**
* No errors were found in the class. Claude pointed out a small typo in the Task 2 intro, and suggested moving the class into its own module (`dice.py`) so that `items.py` can import it in Task 3. The class is kept both in `dice.py` and in the notebook, since it is easier to explain with the code visible.

**What I did myself**
* Wrote the whole `Dice` class: the `__init__` with the check for at least 2 sides, `roll()`, `__repr__` and `__call__`. Also wrote the Task 2 introduction and the test cell, and ran the tests.

---

### Task 3

**Questions about the code**
* *TODO*

**Code written by AI**
* *TODO (remove if none)*

**Testing and discussion**
* *TODO*

**Errors and corrections**
*TODO*

**What I did myself**
*TODO*

---

### Task 4

I used chatGPT to understand how the `super()` function works

---

### Task 5

**Questions about the code**
* *TODO*

**Code written by AI**
* *TODO (remove if none)*

**Testing and discussion**
* *TODO*

**Errors and corrections**
*TODO*

**What I did myself**
*TODO*

---